<a href="https://colab.research.google.com/github/Angelicorp/the-repository-of-micah-delaney-Nbtxai-/blob/main/crimestalker_anonymous.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Crime Analysis Interactive Dashboard

This dashboard integrates the RandomForest predictions, Census poverty data, and EEG stress zones. Use the widgets below to filter and explore the data distributions.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Check if data exists before building widgets
if 'real_tip_df' in globals():
    # Create widgets for interaction
    crime_type_filter = widgets.SelectMultiple(
        options=real_tip_df['crime_type'].unique() if 'crime_type' in real_tip_df.columns else [],
        value=tuple(real_tip_df['crime_type'].unique()) if 'crime_type' in real_tip_df.columns else [],
        description='Crime Type:',
        disabled=False
    )

    threshold_slider = widgets.FloatSlider(
        value=10.0,
        min=0,
        max=100.0,
        step=0.5,
        description='Poverty % >',
        orientation='horizontal'
    )

    output = widgets.Output()

    def update_dashboard(change):
        with output:
            clear_output(wait=True)
            # Filtering logic
            try:
                filtered_df = real_tip_df[
                    (real_tip_df['crime_type'].isin(crime_type_filter.value)) &
                    (real_tip_df['poverty_rate'].astype(float) > threshold_slider.value)
                ]
                print(f"Displaying {len(filtered_df)} points matching criteria.")
                display(filtered_df.head(10))
            except Exception as e:
                print(f"Waiting for data processing to complete... (Error: {e})")

    crime_type_filter.observe(update_dashboard, names='value')
    threshold_slider.observe(update_dashboard, names='value')

    # Display the UI
    ui = widgets.VBox([widgets.HBox([crime_type_filter, threshold_slider]), output])
    display(ui)

    # Trigger initial display
    update_dashboard(None)
else:
    print("Dashboard Ready: Please run the 'Load and Prepare Real Tip Data' cell above and upload your CSV to populate the dashboard.")

# Task
Develop a comprehensive crime analysis system that integrates real tip data from an uploaded CSV file, retrieves actual poverty rates from the Census API, trains a RandomForestRegressor model to predict future crime hotspots, visualizes these predictions, integrates and analyzes EEG stress scores for overlap with crime clusters, and develops interactive dashboard elements to present all analysis results.

## Load and Prepare Real Tip Data

### Subtask:
Replace the synthetic tip data with real tip data by providing functionality to upload a CSV file. The CSV should contain columns such as 'tip_id', 'lat', 'lon', 'crime_type', and 'offender_id'. This step will also include necessary data cleaning and type conversions for the new real tip data.


**Reasoning**:
The subtask requires uploading a CSV file, loading it into a DataFrame, renaming columns, converting data types, handling missing values, and then displaying the head and info of the DataFrame. These steps are interdependent and logically flow into one code block for efficient execution.



In [ ]:
import pandas as pd
from google.colab import files
import io

print("Please upload the CSV file containing real tip data. It should have columns like 'tip_id', 'lat', 'lon', 'crime_type', and 'offender_id'.")

# 1. Upload the CSV file
uploaded = files.upload()

# Check if a file was uploaded
if uploaded:
    for fn in uploaded.keys():
        print(f"User uploaded file '{fn}' with length {len(uploaded[fn])} bytes")

        # 2. Read the uploaded CSV file into a pandas DataFrame
        real_tip_df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
        print("\nOriginal DataFrame head:")
        print(real_tip_df.head())
        print("\nOriginal DataFrame info:")
        real_tip_df.info()

        # 3. Inspect and rename columns
        # Standardize column names to lower case for easier manipulation
        real_tip_df.columns = real_tip_df.columns.str.lower()

        # Define expected columns and map potential user column names
        # This assumes common variations and provides a robust renaming strategy
        column_mapping = {
            'tip_id': 'tip_id',
            'tipid': 'tip_id',
            'id': 'tip_id',
            'latitude': 'lat',
            'longitude': 'lon',
            'crime_type': 'crime_type',
            'crimetype': 'crime_type',
            'type_of_crime': 'crime_type',
            'offender_id': 'offender_id',
            'offenderid': 'offender_id',
            'suspect_id': 'offender_id'
        }

        # Rename columns based on the mapping
        real_tip_df = real_tip_df.rename(columns=column_mapping)

        # Ensure all required columns exist after renaming, if not, fill with None or raise error
        required_columns = ['tip_id', 'lat', 'lon', 'crime_type', 'offender_id']
        for col in required_columns:
            if col not in real_tip_df.columns:
                print(f"Warning: Required column '{col}' not found after renaming. Adding it with NaN values.")
                real_tip_df[col] = pd.NA

        # Select only the required columns and reorder them
        real_tip_df = real_tip_df[required_columns]

        # 4. Convert 'lat' and 'lon' columns to numeric
        real_tip_df['lat'] = pd.to_numeric(real_tip_df['lat'], errors='coerce')
        real_tip_df['lon'] = pd.to_numeric(real_tip_df['lon'], errors='coerce')

        # 5. Check for and handle missing values
        print("\nMissing values before handling:")
        print(real_tip_df.isnull().sum())

        # Drop rows where 'lat' or 'lon' are missing after conversion, as these are critical for spatial analysis
        initial_rows = len(real_tip_df)
        real_tip_df.dropna(subset=['lat', 'lon'], inplace=True)
        rows_dropped = initial_rows - len(real_tip_df)
        if rows_dropped > 0:
            print(f"Dropped {rows_dropped} rows due to missing 'lat' or 'lon' values.")

        # For 'tip_id', 'crime_type', 'offender_id', consider specific handling or drop if critical
        # For now, we'll drop rows with missing tip_id or crime_type as they are essential identifiers
        real_tip_df.dropna(subset=['tip_id', 'crime_type'], inplace=True)
        print("\nMissing values after handling essential columns:")
        print(real_tip_df.isnull().sum())

        # Fill missing offender_id with a placeholder like 'UNKNOWN' if appropriate, or leave as NaN
        # For this task, we'll leave it as NaN as it's not explicitly stated to fill.

        # 6. Display the first 5 rows and info summary of the processed DataFrame
        print("\nProcessed real_tip_df head:")
        print(real_tip_df.head())
        print("\nProcessed real_tip_df info:")
        real_tip_df.info()
else:
    print("No file uploaded. Please re-run the cell and upload a CSV file.")

Please upload the CSV file containing real tip data. It should have columns like 'tip_id', 'lat', 'lon', 'crime_type', and 'offender_id'.


The previous step initiated a file upload prompt. Please upload your CSV file containing the real tip data.

Once the file is successfully uploaded and the cell finishes execution, the code will load the data into `real_tip_df`, rename columns, convert data types, handle missing values, and then display the processed DataFrame.

After the upload and processing are complete, the `real_tip_df` DataFrame will be available for further analysis, and this subtask will be considered finished.

## Integrate Real Poverty Data from Census API

### Subtask:
Retrieve actual poverty rate data from the Census API, and update the `real_tip_df` with real poverty rates based on zip codes.


**Reasoning**:
The first step is to install the `geopy` library, which is necessary for converting latitude and longitude coordinates to zip codes using reverse geocoding.



In [ ]:
pip install geopy
print("geopy library installed successfully.")

**Reasoning**:
The next step is to extract zip codes from the `real_tip_df` by performing reverse geocoding on the latitude and longitude coordinates, creating a new `zip_code` column.



In [ ]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="colab_crime_analysis_app", timeout=10)

def get_zip_code(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True)
        if location and location.address:
            address_components = location.raw.get('address', {})
            zip_code = address_components.get('postcode')
            return zip_code
    except Exception as e:
        print(f"Error geocoding {lat}, {lon}: {e}")
        return None

# Apply the function to create the 'zip_code' column
# To avoid hitting rate limits, especially with a large DataFrame, a small delay is added
# Note: This operation can be very slow for large datasets. Consider optimizing or batching if performance is an issue.
print("Extracting zip codes from latitude and longitude. This may take a while...")
real_tip_df['zip_code'] = real_tip_df.apply(lambda row: get_zip_code(row['lat'], row['lon']), axis=1)

print("Zip code extraction complete.")
print("\nreal_tip_df with new 'zip_code' column head:")
print(real_tip_df.head())
print("\nreal_tip_df info after adding 'zip_code':")
real_tip_df.info()


### Obtain Census API Key

To proceed with retrieving poverty data from the Census API, you will need a personal API key.

**Instructions to obtain your Census API Key:**
1.  Navigate to the official Census Bureau website for developers: [https://api.census.gov/data/key_request.html](https://api.census.gov/data/key_request.html)
2.  Follow the instructions on the website to request and obtain your personal API key.
3.  Once you have your API key, please paste it into the code cell that follows this markdown block. This key will be necessary for accessing the poverty data.

**Reasoning**:
The user has been instructed to obtain a Census API key. Now, a code cell is needed for the user to input their obtained API key so it can be used in subsequent API calls.



In [ ]:
census_api_key = "YOUR_CENSUS_API_KEY_HERE" # Replace with your actual Census API key

if census_api_key == "YOUR_CENSUS_API_KEY_HERE":
    print("Please replace 'YOUR_CENSUS_API_KEY_HERE' with your actual Census API key.")
else:
    print("Census API Key has been set.")

**Reasoning**:
Now that the Census API key is expected to be set, the next step is to define the `get_poverty_data` function as per the instructions, which will be responsible for calling the Census API to retrieve poverty rates for the unique zip codes.



In [ ]:
import requests
import json
import time

def get_poverty_data(zip_codes, api_key, year='2021'):
    """
    Retrieves poverty rate data from the Census API for a list of zip codes.

    Args:
        zip_codes (list): A list of unique zip codes (as strings).
        api_key (str): Your Census API key.
        year (str): The year for which to retrieve data (e.g., '2021').

    Returns:
        pd.DataFrame: A DataFrame with 'zip_code' and 'poverty_rate' columns.
    """
    poverty_data_list = []
    unique_zip_codes = list(set(zip_codes))

    print(f"Fetching poverty data for {len(unique_zip_codes)} unique zip codes...")

    for i, zip_code in enumerate(unique_zip_codes):
        # Census variables for poverty:
        # B17001_001E: Total population for whom poverty status is determined
        # B17001_002E: Population below poverty level
        # The Census API expects 5-digit ZCTA codes, ensure 'zip_code' is formatted correctly.
        if zip_code is None or not str(zip_code).isdigit() or len(str(zip_code)) != 5:
            print(f"Skipping invalid zip code: {zip_code}")
            continue

        # Construct the API URL. We are using ACS 5-year estimates.
        url = f"https://api.census.gov/data/{year}/acs/acs5?get=NAME,B17001_001E,B17001_002E&for=zip code tabulation area:{zip_code}&key={api_key}"

        try:
            response = requests.get(url)
            response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
            data = response.json()

            # The first row is headers, subsequent rows are data
            if len(data) > 1:
                total_population = float(data[1][1]) # B17001_001E
                below_poverty = float(data[1][2]) # B17001_002E

                poverty_rate = (below_poverty / total_population * 100) if total_population > 0 else 0
                poverty_data_list.append({'zip_code': zip_code, 'poverty_rate': poverty_rate})
            else:
                print(f"No data found for zip code: {zip_code} for year {year}")
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for zip code {zip_code}: {e}")
        except (json.JSONDecodeError, IndexError, ValueError) as e:
            print(f"Error parsing data for zip code {zip_code}: {e}")

        # Add a small delay to avoid hitting API rate limits
        time.sleep(0.1)

    return pd.DataFrame(poverty_data_list)

# Ensure the census_api_key is available from the previous cell
if census_api_key == "YOUR_CENSUS_API_KEY_HERE":
    print("Census API Key is not set. Please set your API key in the previous cell.")
    poverty_rates_df = pd.DataFrame(columns=['zip_code', 'poverty_rate'])
else:
    # Get unique zip codes from real_tip_df, filtering out None values and converting to string
    unique_zip_codes = real_tip_df['zip_code'].dropna().astype(str).unique()

    # Call the function to get poverty data
    poverty_rates_df = get_poverty_data(unique_zip_codes, census_api_key)

# Merge the poverty rates with the real_tip_df
initial_rows = len(real_tip_df)
real_tip_df = pd.merge(real_tip_df, poverty_rates_df, on='zip_code', how='left')
merged_rows = len(real_tip_df)

print("\nMerge complete. Rows before merge:", initial_rows, ", rows after merge:", merged_rows)

# Inspect the updated DataFrame
print("\nProcessed real_tip_df with poverty rates head:")
print(real_tip_df.head())
print("\nProcessed real_tip_df info after adding poverty rates:")
real_tip_df.info()

## Prepare Data for Hotspot Prediction Model

### Subtask:
Extract cluster centroids from the previously identified DBSCAN crime clusters. Prepare a dataset suitable for training a machine learning model, incorporating cluster centroids and other relevant features (e.g., historical crime counts, time-series data, poverty rates) to predict future hotspot probabilities or intensity.


**Reasoning**:
The subtask requires several interdependent steps, including importing libraries, selecting and standardizing data, applying DBSCAN clustering, filtering noise, calculating cluster centroids, and aggregating features. Combining these into a single code block ensures a coherent and efficient execution of this data preparation phase.



In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Select the 'lat' and 'lon' columns for clustering
X = real_tip_df[['lat', 'lon']].copy()

# 2. Standardize the selected 'lat' and 'lon' coordinates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Apply DBSCAN clustering
# Experiment with eps and min_samples. These values might need tuning based on data density.
# For example, a small eps like 0.05-0.1 for scaled data often works well for geographic coordinates,
# and min_samples of 5-10 for typical crime hot-spots.
# Let's start with a common setup for scaled data.
db = DBSCAN(eps=0.1, min_samples=10).fit(X_scaled)
real_tip_df['cluster_label'] = db.labels_

print(f"Number of clusters found (including noise -1): {len(np.unique(db.labels_))}")
print("Distribution of cluster labels:")
print(real_tip_df['cluster_label'].value_counts())

# 4. Filter out noise points (clusters labeled -1)
clustered_tips_df = real_tip_df[real_tip_df['cluster_label'] != -1].copy()

print(f"\nRows before filtering noise: {len(real_tip_df)}")
print(f"Rows after filtering noise: {len(clustered_tips_df)}")

# 5. For each remaining cluster, calculate the centroid, crime count, and mean poverty rate
# Ensure poverty_rate is numeric before calculating mean
clustered_tips_df['poverty_rate'] = pd.to_numeric(clustered_tips_df['poverty_rate'], errors='coerce')

cluster_data_for_model = clustered_tips_df.groupby('cluster_label').agg(
    centroid_lat=('lat', 'mean'),
    centroid_lon=('lon', 'mean'),
    crime_count=('tip_id', 'count'),
    mean_poverty_rate=('poverty_rate', 'mean')
).reset_index()

# Rename cluster_label to cluster_id for clarity
cluster_data_for_model.rename(columns={'cluster_label': 'cluster_id'}, inplace=True)

# 6. Display the first 5 rows and info summary of cluster_data_for_model
print("\nCluster data for model head:")
print(cluster_data_for_model.head())
print("\nCluster data for model info:")
cluster_data_for_model.info()

## Train RandomForestRegressor for Hotspot Prediction

### Subtask:
Implement and train a 'RandomForestRegressor' model using the prepared data. The model will aim to predict future crime hotspot locations or intensity, involving data splitting for training and testing, model fitting, and basic evaluation.


**Reasoning**:
The subtask requires importing libraries, defining features and target variables, splitting the data into training and testing sets, initializing and training a RandomForestRegressor model, making predictions, and evaluating the model's performance. These steps are sequential and interdependent, making them suitable for execution within a single code block.



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Define features (X) and target (y)
X = cluster_data_for_model[['centroid_lat', 'centroid_lon', 'mean_poverty_rate']]
y = cluster_data_for_model['crime_count']

# Drop rows with NaN values in X or y, which might occur if 'mean_poverty_rate' had NaNs
X = X.dropna()
y = y[X.index] # Ensure y aligns with cleaned X

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")

# 3. Initialize the RandomForestRegressor model
# Using a random_state for reproducibility
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 4. Train the model
print("\nTraining RandomForestRegressor model...")
model.fit(X_train, y_train)
print("Model training complete.")

# 5. Make predictions on the test set
y_pred = model.predict(X_test)

# 6. Evaluate the model's performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2) Score: {r2:.2f}")

print("\nFirst 10 actual vs predicted values:")
for actual, predicted in list(zip(y_test, y_pred))[:10]:
    print(f"Actual: {actual:.0f}, Predicted: {predicted:.0f}")


# Task
## Visualize Predicted Future Hotspots

### Subtask:
Create visualizations, such as an interactive map or a scatter plot, to display the predicted future crime hotspots, helping to understand potential future crime concentrations.

#### Reasoning:
To effectively visualize crime hotspots, especially predicted ones, an interactive map is highly suitable. `Folium` is chosen for its ability to create interactive leaflet maps directly in a Python notebook. This step will involve combining the test set's geographical coordinates with the model's predictions to display the locations and predicted intensities of future crime hotspots.

```python
import folium

# Combine X_test and y_pred for visualization
predicted_hotspots = X_test.copy()
predicted_hotspots['predicted_crime_count'] = y_pred
predicted_hotspots['actual_crime_count'] = y_test.values # Add actual values for comparison

# Display the first few rows of the predicted_hotspots DataFrame
print("Predicted Hotspots DataFrame Head:")
print(predicted_hotspots.head())

# Calculate the mean latitude and longitude to center the map
mean_lat = predicted_hotspots['centroid_lat'].mean()
mean_lon = predicted_hotspots['centroid_lon'].mean()

# Create a base map centered around the mean coordinates
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=10)

# Add predicted hotspots to the map
print("\nAdding predicted hotspots to the map...")
for idx, row in predicted_hotspots.iterrows():
    # Scale the radius based on predicted crime count.
    # A simple scaling factor is used; this might need tuning.
    # Adding 1 to the count to ensure a minimum radius for even low counts.
    radius = (row['predicted_crime_count'] + 1) * 0.5

    # Create a popup with detailed information
    popup_text = (
        f"<b>Predicted Hotspot</b><br>"
        f"Lat: {row['centroid_lat']:.4f}<br>"
        f"Lon: {row['centroid_lon']:.4f}<br>"
        f"Predicted Crime Count: {row['predicted_crime_count']:.2f}<br>"
        f"Actual Crime Count: {row['actual_crime_count']:.0f}<br>"
        f"Mean Poverty Rate: {row['mean_poverty_rate']:.2f}%"
    )

    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.6,
        popup=popup_text
    ).add_to(m)

# Add actual hotspots from the entire dataset for context (optional, but good for comparison)
# We'll add all actual clusters from 'cluster_data_for_model' to see the overall distribution
print("Adding actual crime clusters (all data) to the map for context...")
for idx, row in cluster_data_for_model.iterrows():
    # Scale radius based on actual crime count
    radius = (row['crime_count'] + 1) * 0.5
    popup_text = (
        f"<b>Actual Crime Cluster</b><br>"
        f"Cluster ID: {row['cluster_id']}<br>"
        f"Lat: {row['centroid_lat']:.4f}<br>"
        f"Lon: {row['centroid_lon']:.4f}<br>"
        f"Actual Crime Count: {row['crime_count']:.0f}<br>"
        f"Mean Poverty Rate: {row['mean_poverty_rate']:.2f}%"
    )
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.3, # Make actual clusters slightly transparent
        popup=popup_text
    ).add_to(m)


print("\nInteractive map generated. Red circles indicate predicted hotspots, blue circles indicate actual historical crime clusters.")
# Display the map
m
```

## Correct DBSCAN Parameters

### Subtask:
Modify the DBSCAN parameters in cell `a5efe204` by changing `min_samples` to a smaller, more appropriate value (e.g., 2) to ensure clusters are formed even with a small number of data points, and re-run the cell. This will address the issue of an empty dataset being passed to the RandomForestRegressor.


### Adjusting DBSCAN Parameters

As instructed, we are modifying the `min_samples` parameter in the DBSCAN clustering step. A `min_samples` value of `10` might be too high for the current dataset, leading to many points being classified as noise or very few clusters being formed. By reducing `min_samples` to `2`, we expect to identify more clusters, even from smaller groups of data points, thus providing more data for the RandomForestRegressor.

After executing the following cell, the `cluster_data_for_model` DataFrame will be re-generated with the new clustering results, which should contain more cluster data for subsequent steps.

**Reasoning**:
As per the subtask, I need to modify the `min_samples` parameter in the DBSCAN clustering in the specified code cell to ensure more clusters are formed, which will help prevent an empty dataset from being passed to the RandomForestRegressor. I will take the code from cell `a5efe204` and update `min_samples` from `10` to `2`.



In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Select the 'lat' and 'lon' columns for clustering
X = real_tip_df[['lat', 'lon']].copy()

# 2. Standardize the selected 'lat' and 'lon' coordinates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Apply DBSCAN clustering
# Experiment with eps and min_samples. These values might need tuning based on data density.
# For example, a small eps like 0.05-0.1 for scaled data often works well for geographic coordinates,
# and min_samples of 5-10 for typical crime hot-spots.
# Let's start with a common setup for scaled data.
db = DBSCAN(eps=0.1, min_samples=2).fit(X_scaled) # Modified min_samples to 2
real_tip_df['cluster_label'] = db.labels_

print(f"Number of clusters found (including noise -1): {len(np.unique(db.labels_))}")
print("Distribution of cluster labels:")
print(real_tip_df['cluster_label'].value_counts())

# 4. Filter out noise points (clusters labeled -1)
clustered_tips_df = real_tip_df[real_tip_df['cluster_label'] != -1].copy()

print(f"\nRows before filtering noise: {len(real_tip_df)}")
print(f"Rows after filtering noise: {len(clustered_tips_df)}")

# 5. For each remaining cluster, calculate the centroid, crime count, and mean poverty rate
# Ensure poverty_rate is numeric before calculating mean
clustered_tips_df['poverty_rate'] = pd.to_numeric(clustered_tips_df['poverty_rate'], errors='coerce')

cluster_data_for_model = clustered_tips_df.groupby('cluster_label').agg(
    centroid_lat=('lat', 'mean'),
    centroid_lon=('lon', 'mean'),
    crime_count=('tip_id', 'count'),
    mean_poverty_rate=('poverty_rate', 'mean')
).reset_index()

# Rename cluster_label to cluster_id for clarity
cluster_data_for_model.rename(columns={'cluster_label': 'cluster_id'}, inplace=True)

# 6. Display the first 5 rows and info summary of cluster_data_for_model
print("\nCluster data for model head:")
print(cluster_data_for_model.head())
print("\nCluster data for model info:")
cluster_data_for_model.info()

**Reasoning**:
As per the subtask, I need to modify the `min_samples` parameter in the DBSCAN clustering in the specified code cell to ensure more clusters are formed, which will help prevent an empty dataset from being passed to the RandomForestRegressor. I will take the code from cell `a5efe204` and update `min_samples` from `10` to `2`.



In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Select the 'lat' and 'lon' columns for clustering
X = real_tip_df[['lat', 'lon']].copy()

# 2. Standardize the selected 'lat' and 'lon' coordinates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Apply DBSCAN clustering
# Experiment with eps and min_samples. These values might need tuning based on data density.
# For example, a small eps like 0.05-0.1 for scaled data often works well for geographic coordinates,
# and min_samples of 5-10 for typical crime hot-spots.
# Let's start with a common setup for scaled data.
db = DBSCAN(eps=0.1, min_samples=2).fit(X_scaled) # Modified min_samples to 2
real_tip_df['cluster_label'] = db.labels_

print(f"Number of clusters found (including noise -1): {len(np.unique(db.labels_))}")
print("Distribution of cluster labels:")
print(real_tip_df['cluster_label'].value_counts())

# 4. Filter out noise points (clusters labeled -1)
clustered_tips_df = real_tip_df[real_tip_df['cluster_label'] != -1].copy()

print(f"\nRows before filtering noise: {len(real_tip_df)}")
print(f"Rows after filtering noise: {len(clustered_tips_df)}")

# 5. For each remaining cluster, calculate the centroid, crime count, and mean poverty rate
# Ensure poverty_rate is numeric before calculating mean
clustered_tips_df['poverty_rate'] = pd.to_numeric(clustered_tips_df['poverty_rate'], errors='coerce')

cluster_data_for_model = clustered_tips_df.groupby('cluster_label').agg(
    centroid_lat=('lat', 'mean'),
    centroid_lon=('lon', 'mean'),
    crime_count=('tip_id', 'count'),
    mean_poverty_rate=('poverty_rate', 'mean')
).reset_index()

# Rename cluster_label to cluster_id for clarity
cluster_data_for_model.rename(columns={'cluster_label': 'cluster_id'}, inplace=True)

# 6. Display the first 5 rows and info summary of cluster_data_for_model
print("\nCluster data for model head:")
print(cluster_data_for_model.head())
print("\nCluster data for model info:")
cluster_data_for_model.info()

## Visualize Predicted Future Hotspots

### Subtask:
Create visualizations, such as an interactive map or a scatter plot, to display the predicted future crime hotspots, helping to understand potential future crime concentrations.


**Reasoning**:
To visualize the predicted hotspots and actual crime clusters, I will generate a code block that uses the `folium` library to create an interactive map. This block will combine the test data with the model's predictions, calculate map coordinates, and add distinct markers for predicted and actual crime hotpsots with relevant information in popups, as per the instructions.



In [ ]:
import folium

# Combine X_test and y_pred for visualization
predicted_hotspots = X_test.copy()
predicted_hotspots['predicted_crime_count'] = y_pred
predicted_hotspots['actual_crime_count'] = y_test.values # Add actual values for comparison

print("Predicted Hotspots DataFrame Head:")
print(predicted_hotspots.head())

# Calculate the mean latitude and longitude to center the map
mean_lat = predicted_hotspots['centroid_lat'].mean()
mean_lon = predicted_hotspots['centroid_lon'].mean()

# Create a base map centered around the mean coordinates
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=10)

# Add predicted hotspots to the map
print("\nAdding predicted hotspots to the map...")
for idx, row in predicted_hotspots.iterrows():
    # Scale the radius based on predicted crime count.
    # Adding 1 to the count to ensure a minimum radius for even low counts.
    radius = (row['predicted_crime_count'] + 1) * 0.5

    # Create a popup with detailed information
    popup_text = (
        f"<b>Predicted Hotspot</b><br>" # Using <b> for bold text
        f"Lat: {row['centroid_lat']:.4f}<br>"
        f"Lon: {row['centroid_lon']:.4f}<br>"
        f"Predicted Crime Count: {row['predicted_crime_count']:.2f}<br>"
        f"Actual Crime Count: {row['actual_crime_count']:.0f}<br>"
        f"Mean Poverty Rate: {row['mean_poverty_rate']:.2f}%"
    )

    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.6,
        popup=folium.Popup(popup_text, max_width=300) # Use folium.Popup for better rendering
    ).add_to(m)

# Add actual hotspots from the entire dataset for context
print("Adding actual crime clusters (all data) to the map for context...")
for idx, row in cluster_data_for_model.iterrows():
    # Scale radius based on actual crime count
    radius = (row['crime_count'] + 1) * 0.5
    popup_text = (
        f"<b>Actual Crime Cluster</b><br>"
        f"Cluster ID: {row['cluster_id']}<br>"
        f"Lat: {row['centroid_lat']:.4f}<br>"
        f"Lon: {row['centroid_lon']:.4f}<br>"
        f"Actual Crime Count: {row['crime_count']:.0f}<br>"
        f"Mean Poverty Rate: {row['mean_poverty_rate']:.2f}%"
    )
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=radius,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.3, # Make actual clusters slightly transparent
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

print("\nInteractive map generated. Red circles indicate predicted hotspots, blue circles indicate actual historical crime clusters.")
# Display the map
m

## Integrate EEG Stress Scores and Analyze Overlap

### Subtask:
Develop functionality to load and process 'EEG stress scores' data. This step will then merge this stress score data with the existing crime cluster information to identify and flag high-stress zones that spatially overlap with crime clusters, exploring potential correlations.


**Reasoning**:
The first step is to create a function to load the EEG stress scores data from a CSV file, including error handling, and then call this function with a placeholder file path to load the data into a DataFrame for initial inspection.



In [ ]:
import pandas as pd
from google.colab import files
import io

def load_eeg_data(file_content, file_name="uploaded_eeg_data.csv"):
    """
    Loads EEG stress scores from uploaded file content into a Pandas DataFrame.

    Args:
        file_content (bytes): The content of the uploaded CSV file.
        file_name (str): The name of the uploaded file.

    Returns:
        pd.DataFrame or None: A DataFrame containing the EEG stress scores
                              if loading is successful, otherwise None.
    """
    try:
        eeg_df = pd.read_csv(io.StringIO(file_content.decode('utf-8')))
        print(f"Successfully loaded EEG data from '{file_name}'.")
        return eeg_df
    except pd.errors.EmptyDataError:
        print(f"Error: The file '{file_name}' is empty.")
        return None
    except pd.errors.ParserError as e:
        print(f"Error: Could not parse '{file_name}'. Check file format. Error: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while loading '{file_name}': {e}")
        return None

print("Please upload the CSV file containing EEG stress scores. It should have columns like 'eeg_id', 'lat', 'lon', and 'stress_score'.")

# 1. Upload the CSV file
uploaded_eeg_files = files.upload()

eeg_df = None
# Check if a file was uploaded
if uploaded_eeg_files:
    for fn in uploaded_eeg_files.keys():
        print(f"User uploaded file '{fn}' with length {len(uploaded_eeg_files[fn])} bytes")
        eeg_df = load_eeg_data(uploaded_eeg_files[fn], fn)
        break # Assuming only one file is uploaded at a time
else:
    print("No file uploaded. Please re-run the cell and upload a CSV file.")

if eeg_df is not None:
    print("\nHead of EEG Stress Scores DataFrame:")
    print(eeg_df.head())
    print("\nInfo of EEG Stress Scores DataFrame:")
    eeg_df.info()
